[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/semantica-agi/semantica/blob/main/cookbook/advanced/12_Unstructured_to_Ontology.ipynb)

> **中文译文** —— 由社区翻译，非上游原文；源文件变更后以英文原版为准。步骤间的逻辑导读为译者补充。

# 非结构化文本到本体（Unstructured Text to Ontology）

欢迎来到从非结构化文本抽取结构化本体的高级指南。本 notebook 探索 Semantica 的两条强大路线：

1.  **经典 NLP 流水线**：命名实体识别（NER）+ 关系抽取
2.  **生成式 AI 流水线**：用大语言模型（LLM）直接做概念建模

我们将对比两条路线、可视化结果、并校验生成的本体。

### 先看主线

14 篇讲过本体生成（LLM 只占一节），这一篇把两条路线摆上擂台——**同一篇文本，两法各跑**：NLP 是**自底向上**（实体 → 关系 → 本体：确定性、可溯源、离线可用），LLM 是**自顶向下**（读文本直接提 schema：上下文感知、能处理歧义、但非确定、要 API）。三段：

**备料**：一段描述科技公司的富文本（QuantumDynamics），两条路线的同一考题。

**双线对决（路线 1、路线 2）**：NLP 流水线五步走；LLM 一把梭。

**对比与出口**：并排可视化看结构差异——**NLP 偏字面、LLM 偏概念**；择优（或经 `ReuseManager` 合并）导出 OWL。

**文档**：[API 参考](https://semantica.readthedocs.io/reference/ontology/）

## 环境与安装

确认装好 Semantica 及全部依赖。

In [ ]:
!pip install -qU semantica

In [ ]:


from semantica.utils.logging import get_logger

logger = get_logger("unstructured_guide")
print("Environment setup complete.")

---

## 输入文本

用一段描述科技公司的富文本测两条抽取路线。

In [ ]:
text_corpus = """
QuantumDynamics is a leading AI research lab founded by Dr. Elena Rostova in 2018. 
The lab is headquartered in Zurich, Switzerland, and focuses on quantum computing algorithms. 
Dr. Rostova serves as the Chief Scientist. 
The lab has released products like the Q-1 Processor and the NeuralBridge SDK. 
QuantumDynamics collaborates with major universities such as MIT and ETH Zurich.
"""

---

## 路线 1：经典 NLP 流水线

这条路自底向上建本体：
1.  **抽实体**：识别名词/专有名词（如 "QuantumDynamics"、"Zurich"）
2.  **抽关系**：识别连接它们的动词（如 "headquartered in"）
3.  **生成本体**：把三元组映射成类与属性

**优点**：确定性、可溯源、离线可用。
**缺点**：受底层 NLP 模型词表和灵活性的限制。

In [ ]:
from semantica.semantic_extract import NERExtractor, RelationExtractor
from semantica.ontology import OntologyGenerator, OntologyOptimizer

# 1. Initialize Extractors
ner = NERExtractor()
re = RelationExtractor()

# 2. Extract Entities
print("Extracting entities...")
entities = ner.extract(text_corpus)

# Note: entities are returned as Entity objects (dataclasses), not dictionaries.
# We access properties using dot notation (e.g., entity.text, entity.label).
print(f"Found {len(entities)} entities.")
for e in entities[:5]:
    print(f" - {e.text} ({e.label}) [Conf: {e.confidence}]")

# 3. Extract Relationships
print("\nExtracting relationships...")
relationships = re.extract(text_corpus, entities)

# Note: relationships are returned as Relation objects.
print(f"Found {len(relationships)} relationships.")
for r in relationships:
    print(f" - {r.subject.text} -> {r.predicate} -> {r.object.text}")

# 4. Prepare Data for Ontology Generation
# The OntologyGenerator expects dictionaries, so we convert our objects.
# We also ensure we handle both object attributes and potential dictionary keys for robustness.
entities_data = []
for e in entities:
    if hasattr(e, 'to_dict'):
        entities_data.append(e.to_dict())
    else:
        # Manual conversion for dataclasses without to_dict
        entities_data.append({
            "id": getattr(e, "text", str(e)),
            "text": getattr(e, "text", str(e)),
            "type": getattr(e, "label", getattr(e, "type", "Unknown")),
            "confidence": getattr(e, "confidence", 1.0)
        })

relationships_data = []
for r in relationships:
    if hasattr(r, 'to_dict'):
        relationships_data.append(r.to_dict())
    else:
        # Manual conversion for dataclasses without to_dict
        # Handle nested Entity objects in subject/object fields
        subj = r.subject
        obj = r.object
        subj_text = getattr(subj, "text", str(subj))
        obj_text = getattr(obj, "text", str(obj))
        
        relationships_data.append({
            "source": subj_text,
            "target": obj_text,
            "type": getattr(r, "predicate", getattr(r, "type", "related_to")),
            "confidence": getattr(r, "confidence", 1.0)
        })

# 5. Generate Structure
generator = OntologyGenerator()
nlp_ontology = generator.generate_ontology(
    {"entities": entities_data, "relationships": relationships_data},
    name="QuantumOntologyNLP"
)

# 6. Optimize (Clean up)
optimizer = OntologyOptimizer()
nlp_ontology = optimizer.optimize_ontology(nlp_ontology, remove_redundancy=True)

print(f"\nGenerated NLP Ontology with {len(nlp_ontology['classes'])} classes and {len(nlp_ontology['properties'])} properties.")

---

## 路线 2：生成式 AI 流水线（LLM）

这条路用大语言模型"读"文本，直接提出 schema。

**优点**：上下文感知、能处理歧义、生成类人风格的类名。
**缺点**：非确定性、需要 API 访问。

*注：这一步需要配好 LLM 提供商（如 OpenAI）。*

In [ ]:
from semantica.ontology import LLMOntologyGenerator

try:
    # Initialize LLM Generator (ensure OPENAI_API_KEY is set in env)
    llm_gen = LLMOntologyGenerator(provider="openai", model="gpt-4")
    
    print("Generating ontology with LLM...")
    llm_ontology = llm_gen.generate_ontology_from_text(
        text=text_corpus,
        name="QuantumOntologyLLM"
    )
    
    print(f"Generated LLM Ontology with {len(llm_ontology['classes'])} classes and {len(llm_ontology['properties'])} properties.")
    print("Classes detected:", [c['name'] for c in llm_ontology['classes']])
    
except Exception as e:
    print(f"Skipping LLM generation: {e}")
    llm_ontology = None

---

## 用可视化对比结果

把两个本体并排可视化（如果都有），看结构差异——**NLP 模型偏字面，LLM 模型偏概念**。

In [ ]:
from semantica.visualization import OntologyVisualizer

visualizer = OntologyVisualizer()

print("--- NLP Approach Visualization ---")
fig_nlp = visualizer.visualize_structure(nlp_ontology, output="interactive")
if fig_nlp: fig_nlp.show()

if llm_ontology:
    print("--- LLM Approach Visualization ---")
    fig_llm = visualizer.visualize_structure(llm_ontology, output="interactive")
    if fig_llm: fig_llm.show()

---

## 导出为 OWL

最后，选出更好的那个（或用 `ReuseManager` 合并两者——其他指南有讲），导出。

In [ ]:
from semantica.export import OWLExporter

exporter = OWLExporter()

# Export the NLP ontology by default, or the LLM one if preferred
target_ontology = llm_ontology if llm_ontology else nlp_ontology

output_file = "quantum_ontology.ttl"
exporter.export(target_ontology, output_file, format="turtle")
print(f"Successfully exported ontology to {output_file}")

## 小结

### 回顾主线

两条路线过完招：NLP **自底向上**——实体、关系、三元组步步为营，确定性可溯源；LLM **自顶向下**——通读全文直接成 schema，概念化但有随机性。没有万能冠军：要可审计走 NLP，要概念覆盖走 LLM，讲究的用法是两路都跑、对比择优。

你学会了：
1.  **程序化抽取本体**：用 `NERExtractor` 做可靠的数据驱动建模
2.  **用 AI 生成本体**：用 `LLMOntologyGenerator` 做概念化、高层建模
3.  **可视化与对比**：用 `OntologyVisualizer` 检视结构差异
4.  **校验与导出**：存成 OWL 标准前保证质量